# Week 1 Day 5 Lab 2 (free version) - Harness Engineering

## Deep Agents works the queue on its own

Deep Agents is LangChain's ready-made harness. It's built on `create_agent`, the same function we used on Day 2, with middleware added for file tools, subagents, a to-do list and summarizing a long conversation. It runs on any model LangChain supports, so we can use OpenRouter.

This is the same job as Lab 1: a folder with three claims from the queue, the policy wording and the claims manual, and the instruction to decide every claim. We don't write any tools.

## 4 parts:

PART 1: The claims desk

PART 2: Setting up the harness

PART 3: Working the queue

PART 4: How it did

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This version uses the free version of dots.3 Note Preview on OpenRouter, so it costs nothing. We name one free model because <code>openrouter/free</code> picks a different model for nearly every call, and some of them can't keep a job like this on track.<br/><br/>
            An OpenRouter account without credit can make 50 free requests a day. Three claims take about 35 requests and a few minutes, so you can run this once a day. If you have bought credit, the main version in <a href="week1day5_lab2.ipynb">week1day5_lab2.ipynb</a> costs about 3 cents.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
from pathlib import Path

import pandas as pd
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import ModelRetryMiddleware, TodoListMiddleware
from langchain_openai import ChatOpenAI

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

OpenRouter API key found


## Part 1: The claims desk

The claims desk is the same as in Lab 1. We use the first three of the five claims marked for the tiny version of the queue: an approval, a decline and a referral.

In [3]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [4]:
claims, labels = get_sample_data(tiny=True)
claims = claims[:3]
labels = {claim["claim_id"]: labels[claim["claim_id"]] for claim in claims}

for claim in claims:
    print(claim["claim_id"], claim["summary"])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples


In [5]:
DESK = Path("lab2free_desk")


def make_desk(claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (DESK / folder).mkdir(parents=True)
    for claim in claims:
        (DESK / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (DESK / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (DESK / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, DESK / name)


make_desk(claims)
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

## Part 2: Setting up the harness

The instructions are the same as in Lab 1, apart from calling the task list a to-do list, which is Deep Agents' name for it.

In [6]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a to-do list with one item for each claim, and give each claim to the claim-handler subagent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Deep Agents</h2>
            <span style="color:#118AB2;"><code>create_deep_agent</code> gives the agent file tools (<code>ls</code>, <code>read_file</code>, <code>write_file</code>, <code>edit_file</code>, <code>glob</code>, <code>grep</code>) and <code>task</code>, which hands work to a subagent.<br/><br/>
            The to-do list is an extra since version 0.7, so we add it with LangChain's <code>TodoListMiddleware</code>, which gives the agent a <code>write_todos</code> tool.<br/><br/>
            <code>FilesystemBackend</code> points the file tools at the claims desk. With <code>virtual_mode=True</code> the agent sees the desk as <code>/</code> and can't reach anything outside it.
            </span>
        </td>
    </tr>
</table>

In [7]:
# Substituted the local qwen3.5:4b model for dots-studio/dots-3-note-preview:free
# because the OpenRouter free-tier daily quota is exhausted (resets next day).
llm = ChatOpenAI(
    model="qwen3.5:4b",
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=120,
    max_retries=5,
)

Free models are sometimes busy and a call can fail. LangChain's `ModelRetryMiddleware` waits a moment and tries a failed call again. We give it to the main agent and to the subagent.

In [8]:
claim_handler = {
    "name": "claim-handler",
    "description": "Decides one claim and writes its decision file. Give it the claim id.",
    "system_prompt": HANDLER_INSTRUCTIONS,
    "middleware": [ModelRetryMiddleware(max_retries=3)],
}

agent = create_deep_agent(
    model=llm,
    system_prompt=INSTRUCTIONS,
    subagents=[claim_handler],
    middleware=[TodoListMiddleware(), ModelRetryMiddleware(max_retries=3)],
    backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
)

## Part 3: Working the queue

As the agent works we print each step and count the model calls. The stream gives us an update after each step of the agent, and the updates from the `model` step hold the model's replies. The subagent's steps are indented.

In [9]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call["args"]
    if not isinstance(args, dict):  # a free model sometimes sends its arguments as one string
        return tool_call["name"]
    if "file_path" in args or "path" in args:
        return f"{tool_call['name']} {args.get('file_path') or args.get('path')}"
    if tool_call["name"] == "task":
        return f"task: {args.get('description', '').splitlines()[0][:80]}"
    if tool_call["name"] == "write_todos":
        return "write_todos: " + ", ".join(f"{todo['content'][:30]} ({todo['status']})" for todo in args.get("todos", []))
    return tool_call["name"]

In [10]:
calls = 0

for namespace, update in agent.stream(
    {"messages": [{"role": "user", "content": "Please work the queue."}]},
    stream_mode="updates",
    subgraphs=True,
    config={"recursion_limit": 200},
):
    if "model" in update:
        indent = "      " if namespace else ""
        for message in update["model"]["messages"]:
            calls += 1
            for tool_call in message.tool_calls:
                print(indent + describe(tool_call))
            if not namespace and message.text.strip():
                print(message.text.strip())

print(f"\nFinished after {calls} model calls")

write_todos: Explore the folder structure a (in_progress), Process all claims in claims/ (pending), Write handover.md with summary (pending)


ls /
ls /claims
ls /policies
ls /customers


read_file /claims/CLM-1007.json
read_file /claims/CLM-1015.json
read_file /claims/CLM-1017.json
read_file /policy_wording.md
read_file /claims_manual.md
read_file /company.json


read_file /policies/POL-2007.json
read_file /policies/POL-2015.json
read_file /policies/POL-2017.json
read_file /customers/CUS-3007.json
read_file /customers/CUS-3015.json
read_file /customers/CUS-3017.json


task: Process claim CLM-1007: Baggage damage from phone dropped in shower. Check polic


      read_file /claims/CLM-1007.json
      read_file /policies/POLICY.json
      read_file /customers/CUST-12345.json
      read_file /policy_wording.md
      read_file /claims_manual.md


... (20 outputs left out)

write_todos: Explore the folder structure a (completed), Process all claims in claims/ (completed), Write handover.md with summary (in_progress)


write_file /handover.md


write_todos: Explore the folder structure a (completed), Process all claims in claims/ (completed), Write handover.md with summary (completed)



Finished after 33 model calls


## Part 4: How it did

We score the decision files with the same test as in Lab 1.

In [11]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get("decision") == label["decision"] and abs(float(answer.get("payout") or 0) - label["payout"]) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / "decisions" / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
    rows.append({
        "claim": claim_id,
        "decision": answer.get("decision"),
        "payout": answer.get("payout"),
        "handler's decision": label["decision"],
        "handler's payout": label["payout"],
        "right": is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

3 of 3 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,approve,50.0,approve,50.0,True
1,CLM-1015,decline,0.0,decline,0.0,True
2,CLM-1017,refer,0.0,refer,0.0,True


And here's the note it left for the handler.

In [12]:
display(Markdown((DESK / "handover.md").read_text(encoding="utf-8")))

# Claim Handling Summary

## Claims Processed

| Claim ID | Customer | Plan | Decision | Payout | Notes |
|----------|----------|------|----------|--------|-------|
| CLM-1007 | Greg Kowalski | Economy | Approve | $50.00 | Phone liquid damage covered - repair shop report confirms beyond economical repair. Per-item limit ($150) minus $100 deductible = $50. |
| CLM-1015 | Kate Callister | First | Decline | $0.00 | Not a covered reason for trip interruption. Oversleeping/child left home is not listed in section 2 of policy wording. |
| CLM-1017 | Emily Nguyen | Economy | Refer | $0.00 | Police report missing - required for theft claims per policy section 5 (must be made within 24 hours). Customer states they'll provide it when found. |

## Key Decisions and Rationales

### CLM-1007 - Baggage Damage
**Coverage determination**: Accidental damage to personal effects is covered under Section 5 of the policy wording. The phone was dropped in a hotel shower (liquid damage), not unattended/abandoned items which are excluded. 

**Documents verified**: 
- Repair shop report confirms phone sustained extensive liquid damage and is beyond economical repair
- Replacement receipt provides proof of value at $1,099

**Payment calculation**: Per-item limit for Economy plan is $150. With $100 deductible for Baggage and Personal Effects: $150 - $100 = $50 payout. This requires no sign-off (under $1,000 threshold).

### CLM-1015 - Trip Interruption
**Coverage determination**: The incident occurred when the customer overslept and forgot their 8-year-old son Kevin on a family trip to Paris. This reason is not covered under Section 2 of the policy wording. Covered reasons for trip interruption are: (1) Sickness/injury/death confirmed in writing; (2) Severe weather/natural disaster stopping carrier service 24+ hours; (3) Home/destination made uninhabitable by fire/flood/storm; (4) Jury duty or subpoena; (5) Job loss through no fault after one year. Oversleeping is not a covered reason.

**No add-ons affect coverage** - this claim falls outside the policy's covered reasons entirely.

### CLM-1017 - Baggage Theft
**Coverage determination**: Theft of baggage and personal effects during the covered trip (July 14, Naples) is covered under Section 5 of the policy wording. However, the claim cannot be approved or declined until all required documents are present.

**Missing document**: A police report made within 24 hours of the theft is **required** per Section 5: "Theft must be reported to the local police within 24 hours, and we need a copy of the police report." The customer stated they filed a report but it hasn't been provided yet.

**Action required**: Refer this claim for customer follow-up requesting the police report within the 30-day timeframe specified in the Claims Manual. Once the report is received, reassess whether to approve or decline (other elements like proof of value appear complete with purchase receipts).

## Notes for Handler

1. **Customer contacts**:
   - CLM-1007: Greg Kowalski - approved claim, notify customer payment will be issued ($50)
   - CLM-1015: Kate Callister - declined claim due to non-covered reason, explain that oversleeping is not a covered trip interruption reason per Section 2
   - CLM-1017: Emily Nguyen - request police report within 30 days

2. **Policy reminders**:
   - For accidental damage claims, a repair estimate or statement of irreparability must be provided (verified for CLM-1007)
   - Theft requires police report within 24 hours (missing in CLM-1017)
   - Trip interruption only covers reasons listed in Section 2 - customer's story must align with one of the five specific reasons

3. **Payout thresholds**: 
   - $50 payout for CLM-1007 does not require sign-off (under $1,000 threshold)
   - No pending payouts requiring handler sign-off at this time


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Build or buy</h2>
            <span style="color:#118AB2;">Deep Agents is open source and runs on any model, so you can pick a cheap one and read exactly what the harness does. The Claude Agent SDK gives you Claude Code's harness, tuned for Claude models.<br/><br/>
            Next we build the same pieces into UnderCover ourselves: a to-do list, tools to read the manual, and a subagent for each claim.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Look back through the steps and find where the agent used its to-do list and where it handed work to the subagent.<br/><br/>
            Then try other free models from OpenRouter's list, or <code>openrouter/free</code>, and see which ones can work the queue and how their answers compare.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

## Solution - assignment challenge

**Was this experiment run? No.** No new harness run was made while writing this solution: OpenRouter's free quota is exhausted, and a full harness run is slow and uses many requests. The cells below are ready to run; each variant catches errors, so a failed model does not wipe out earlier results.

### What to change and what to measure

The harness rebuilds an agent for each variant, works the same five claims on a fresh claims desk, and compares the decision files with the handlers' answers using `is_right`. We record:

* **accuracy** — how many of the five claims are right on decision and payout;
* **cost** — from `token_usage["cost"]` when the provider reports it (Ollama is local, so its cost is $0; add your own electricity or GPU rental cost if that matters);
* **time** — wall-clock seconds for the whole run, and the number of model calls.

A single run of five claims is a small sample: one wrong decision moves accuracy by 20 points. Repeat a promising variant a few times before trusting the difference, and beware that the agent can take a different tool path each run.

In [ ]:
# Lab 1 has no OpenRouter connection, so supply a default for the local-only runs.
OPENROUTER_BASE_URL = globals().get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
openrouter_api_key = globals().get("openrouter_api_key")


def run_variant(model_name, effort=None, base_url=None, api_key=None, label=None):
    """Rebuild the harness for one model/effort, work the queue, score it and time it."""
    make_desk(claims)  # fresh desk: old decision files must not leak into this run
    kwargs = {
        "model": model_name,
        "base_url": base_url or OPENROUTER_BASE_URL,
        "api_key": api_key or openrouter_api_key,
        "max_tokens": 4000,  # this account's allowance rejects the 131072 default
        "timeout": 120,
        "max_retries": 5,
    }
    if effort:
        kwargs["reasoning_effort"] = effort
    variant_llm = ChatOpenAI(**kwargs)
    variant_agent = create_deep_agent(
        model=variant_llm,
        system_prompt=INSTRUCTIONS,
        subagents=[claim_handler],
        middleware=[TodoListMiddleware(), ModelRetryMiddleware(max_retries=5, initial_delay=5, max_delay=120)],
        backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
    )

    start = time.time()
    calls = tokens = cost = 0
    for namespace, update in variant_agent.stream(
        {"messages": [{"role": "user", "content": "Please work the queue."}]},
        stream_mode="updates", subgraphs=True, config={"recursion_limit": 200},
    ):
        if "model" in update:
            for message in update["model"]["messages"]:
                calls += 1
                usage = (message.response_metadata or {}).get("token_usage") or {}
                tokens += (usage.get("prompt_tokens") or 0) + (usage.get("completion_tokens") or 0)
                cost += usage.get("cost") or 0.0
    seconds = round(time.time() - start, 1)

    right = 0
    for claim_id, handler in labels.items():
        path = DESK / "decisions" / f"{claim_id}.json"
        answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
        right += is_right(answer, handler)
    return {
        "variant": label or f"{model_name} ({effort or 'default'})",
        "decisions right": f"{right}/{len(labels)}",
        "accuracy (%)": round(100 * right / len(labels)),
        "model calls": calls,
        "tokens": tokens,
        "cost ($)": round(cost, 4),
        "seconds": seconds,
    }


def compare_variants(variants):
    """variants: (model_name, effort, label, base_url, api_key) tuples."""
    rows = []
    for model_name, effort, label, base_url, api_key in variants:
        try:
            rows.append(run_variant(model_name, effort, base_url, api_key, label))
            print(f"Done: {label or model_name}")
        except Exception as error:
            print(f"Not run: {label or model_name}: {type(error).__name__}: {error}")
            rows.append({"variant": label or model_name, "error": f"{type(error).__name__}: {error}"})
    return pd.DataFrame(rows)

### Free-version variants

The free daily quota allows about one run of this notebook, so plan the models you want to test before you spend it. `openrouter/free` routes each call to a different free model, which makes a fair comparison hard; prefer a named free model (id ending in `:free`) and check that it supports Structured Outputs. Because the quota is exhausted here, the example list starts with the local model this run substituted. Add free ids from openrouter.ai/models when the quota resets.

The runner below takes both an OpenRouter base URL and a local Ollama URL, so you can compare free models and local models in the same table. Cost is $0 for both; the interesting columns are accuracy, calls and seconds.

In [ ]:
LAB2FREE_VARIANTS = [
    ("qwen3.5:4b", None, "local substitution used in this run", "http://localhost:11434/v1", "ollama"),
    ("dots-studio/dots-3-note-preview:free", None, "named free model", OPENROUTER_BASE_URL, openrouter_api_key),
    ("openrouter/free", None, "routed free model", OPENROUTER_BASE_URL, openrouter_api_key),
]

lab2free_comparison = compare_variants(LAB2FREE_VARIANTS)
print(lab2free_comparison.to_string(index=False))